# Capstone · Notebook 1. Pretraining

*Modern LLMs course · capstone · edition 2026.10.3*

> **What happens here.** We walk the whole road from raw documents to a trained
> course reference language model: the corpus cleaning pipeline (M9), byte-level
> BPE from scratch (M2), sharding the token stream, the 20× Chinchilla-style reference budget
> arithmetic (M9), and the run itself with Muon and a WSD schedule (M9).
>
> **Assumed background.** Modules 2–9. Module 1 helps but is not required — at
> the end we read our own model with the same passport approach it teaches.
>
> **The Colab contract.** The mandatory path runs on CPU and needs no network:
> by default the corpus is synthetic and generated right here. Downloading real
> data is a separate cell behind a flag that is off. No number in this notebook
> depends on a network call.

**Profiles.** `SMOKE` (the default) is a short end-to-end mechanism check.
`FULL` uses profile M and the 20× Chinchilla-style reference budget. Runtime is
hardware- and software-dependent: `auto` selects BF16 on capable GPUs, FP16 with
gradient scaling on older CUDA devices such as T4, and FP32 on CPU.

On a first pass, run the cells in order and connect the code to the explanations. On a second, change one setting, retain the results, and state which claim the comparison can test.

## 1. Environment

In [ ]:
import importlib
import sys
import subprocess
import zipfile
from pathlib import Path

ARCHIVE_GLOB = "modern-llms-capstone-*.zip"

def _activate_capstone_root(candidate: Path):
    package_init = candidate / "minillm" / "__init__.py"
    if not package_init.exists():
        return None
    root = str(candidate.resolve())
    if root not in sys.path:
        sys.path.insert(0, root)
    return root

def _find_capstone_root(search_root: Path):
    direct = _activate_capstone_root(search_root)
    if direct:
        return direct
    for child in sorted(search_root.glob("modern-llms-capstone-*")):
        found = _activate_capstone_root(child)
        if found:
            return found
    return None

def _safe_extract(archive_path: Path, destination: Path):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise RuntimeError(f"unsafe path in archive: {member.filename}")
        archive.extractall(destination)

def ensure_minillm():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        found = _find_capstone_root(candidate)
        if found:
            return "project package: " + found

    try:
        import minillm  # noqa: F401
        return "minillm is already importable"
    except ImportError:
        pass

    archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))
    if not archives:
        try:
            from google.colab import files
        except ImportError:
            files = None
        if files is not None:
            print("Upload the Capstone ZIP package from the Stepik lesson. The directory structure will be restored automatically.")
            files.upload()
            archives = sorted(Path.cwd().glob(ARCHIVE_GLOB))

    if archives:
        archive_path = archives[-1]
        destination = Path.cwd() / ".capstone_runtime"
        destination.mkdir(exist_ok=True)
        _safe_extract(archive_path, destination)
        matches = sorted(destination.glob("*/minillm/__init__.py"))
        if len(matches) != 1:
            raise RuntimeError(
                f"expected one minillm package in {archive_path.name}, "
                f"found {len(matches)}"
            )
        root = _activate_capstone_root(matches[0].parent.parent)
        if root:
            return "project archive: " + str(archive_path)

    raise RuntimeError("minillm was not found. Download the language-specific Capstone ZIP package from the Stepik lesson and either extract it in full or upload the ZIP to the current Colab session.")

print(ensure_minillm())

for package in ("torch", "numpy", "matplotlib", "regex"):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package],
            check=True,
        )

import os, torch, numpy as np, hashlib
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# ------------------------------------------------------------------ run profile
PROFILE = "SMOKE"          # "SMOKE" | "FULL"
USE_REAL_CORPUS = False    # network download of real data — see §3
SEED = 0

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from minillm.precision import resolve_compute_dtype
COMPUTE_DTYPE = resolve_compute_dtype(DEVICE, "auto")

SETTINGS = {
    "SMOKE": dict(docs=8_000, vocab_size=2_048, seq_len=128,
                  model_profile="S", steps=250, micro_batch_size=8, grad_accum=1),
    "FULL":  dict(docs=200_000, vocab_size=16_384, seq_len=512,
                  model_profile="M", steps=18_886, micro_batch_size=16, grad_accum=4),
}[PROFILE]
if os.getenv("CAPSTONE_FAST_VALIDATION") == "1":
    SETTINGS = {**SETTINGS, "docs": min(SETTINGS["docs"], 800),
                "vocab_size": min(SETTINGS["vocab_size"], 512),
                "steps": min(SETTINGS["steps"], 12),
                "micro_batch_size": min(SETTINGS["micro_batch_size"], 2),
                "grad_accum": 1}
print(PROFILE, "on", DEVICE, "with", COMPUTE_DTYPE, "compute ->", SETTINGS)

## 2. The corpus and its cleaning funnel

The order of pipeline stages is not arbitrary — it runs cheap to expensive
(M9). Heuristics cost a pass over a line, dedup costs hashing, a model-based
quality classifier costs inference. So the classifier only ever sees what the
earlier stages let through, and the pipeline's price is set by those.

The synthetic corpus below is assembled from a slot grammar in two languages and
deliberately spoiled: it contains duplicates, near-duplicates, documents that are
too short, and symbol junk. That is the raw material a funnel needs — on clean
data there is nothing to demonstrate.

In [ ]:
import random
from minillm.corpus import FilterConfig, pipeline

RU_SLOTS = dict(
    subject=["языковая модель", "трансформер", "механизм внимания", "токенизатор",
             "оптимизатор", "нормализация", "остаточная связь", "роутер экспертов",
             "кэш ключей", "эмбеддинг", "спекулятивный декод", "квантизация"],
    verb=["предсказывает", "обрабатывает", "сокращает", "ускоряет", "стабилизирует",
          "ограничивает", "восстанавливает", "перераспределяет", "измеряет"],
    obj=["следующий токен", "длинный контекст", "стоимость шага", "объём памяти",
         "масштаб активаций", "разброс градиентов", "поток данных", "качество ответа",
         "нагрузку экспертов", "пропускную способность"],
    tail=["на длинных последовательностях", "при обучении в низкой точности",
          "без потери качества", "за счёт дополнительной матрицы",
          "ценой роста числа параметров", "в фазе декодирования",
          "по сравнению с базовой схемой", "при неизменном бюджете вычислений"],
)
EN_SLOTS = dict(
    subject=["a language model", "the transformer", "the attention mechanism",
             "the tokenizer", "the optimizer", "normalization", "the residual stream",
             "the expert router", "the key cache", "the embedding table",
             "speculative decoding", "quantization"],
    verb=["predicts", "processes", "reduces", "accelerates", "stabilizes", "bounds",
          "restores", "redistributes", "measures"],
    obj=["the next token", "a long context", "the cost per step", "the memory footprint",
         "the scale of activations", "the spread of gradients", "the data stream",
         "the quality of the answer", "the load across experts", "the throughput"],
    tail=["on long sequences", "when training in low precision", "without losing quality",
          "at the price of one more matrix", "at the cost of more parameters",
          "during the decode phase", "compared with the baseline scheme",
          "under an unchanged compute budget"],
)


def sentence(rng: random.Random, slots: dict) -> str:
    parts = [rng.choice(slots["subject"]), rng.choice(slots["verb"]),
             rng.choice(slots["obj"]), rng.choice(slots["tail"])]
    return " ".join(parts).capitalize() + "."


def synthetic_corpus(n_docs: int, seed: int = 0) -> list[str]:
    """A bilingual corpus from a slot grammar, with junk mixed in on purpose.

    This is no pretraining corpus: one grammar, a narrow lexicon. Its job is to
    give the pipeline and the tokenizer reproducible, unambiguously licence-clean
    material where every mechanism is visible. The slots supply combinatorial
    variety — without it near-dup dedup would rightly collapse the whole corpus.
    """
    rng = random.Random(seed)
    docs = []
    for i in range(n_docs):
        slots = RU_SLOTS if i % 2 else EN_SLOTS
        docs.append(" ".join(sentence(rng, slots) for _ in range(rng.randint(3, 6))))
    # Junk for the funnel: exact copies, near-copies, short and symbol-only.
    docs += docs[:n_docs // 20]                                   # exact duplicates
    docs += [d + " Addendum." for d in docs[:n_docs // 40]]       # near-duplicates
    docs += ["ok"] * (n_docs // 40)                               # too short
    docs += ["!@#$%^&*()" * 8] * (n_docs // 40)                   # symbol junk
    rng.shuffle(docs)
    return docs

raw_docs = synthetic_corpus(SETTINGS["docs"], seed=SEED)
print(f"raw documents: {len(raw_docs):,}")

In [ ]:
clean_docs, report = pipeline(raw_docs, FilterConfig(min_chars=60),
                              near_dup_threshold=0.85)
print(report)
print(f"\nresult: {len(clean_docs):,} documents, "
      f"{sum(len(d) for d in clean_docs):,} characters")

# Split documents BEFORE tokenizer training: validation contributes neither
# vocabulary entries nor BPE merges. Global cleaning happened before the split,
# so exact and near duplicates cannot straddle train and validation.
split_rng = random.Random(SEED + 17)
split_rng.shuffle(clean_docs)
valid_count = max(1, round(0.05 * len(clean_docs)))
valid_docs = clean_docs[:valid_count]
train_docs = clean_docs[valid_count:]
print(f"document split: train={len(train_docs):,}, valid={len(valid_docs):,}")

Read the funnel as a record of selection. A high rejection rate could reflect dirty input, an overly strict threshold, or both. Inspect retained and rejected examples near each threshold, then compare languages and document types. Here the corruptions were inserted deliberately, so the rejection reasons can be checked against the generator rather than judged from the survival rate alone.

## 3. Real data (optional)

The mandatory path needs no real data and takes no published number from it.
The cell below is enabled by the `USE_REAL_CORPUS` flag from §1.

Options whose licences we verified:

| Corpus | Licence | Note |
|---|---|---|
| `HuggingFaceFW/fineweb-2`, subset `rus_Cyrl` | ODC-By 1.0 | attribution required; CommonCrawl terms of use also apply |
| `roneneldan/TinyStories` | CDLA-Sharing-1.0 | synthetic stories with a narrow lexicon; share-alike covers derived **datasets**, not a trained model |

Model size alone does not determine coherence. TinyStories offers a narrower task than broad web text; outcomes also depend on training budget, coverage, and evaluation. Compare held-out loss and actual continuations under a recorded protocol. The optional data path is an extension, not a required download.

In [ ]:
if USE_REAL_CORPUS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset

    DATA_REVISIONS = {
        "fineweb2": "v2.1.1",
        "tinystories": "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64",
    }
    train_target = SETTINGS["docs"] // 2
    eval_target = max(100, SETTINGS["docs"] // 40)

    ru_train = load_dataset("HuggingFaceFW/fineweb-2", name="rus_Cyrl",
                            split="train", streaming=True,
                            revision=DATA_REVISIONS["fineweb2"])
    ru_eval = load_dataset("HuggingFaceFW/fineweb-2", name="rus_Cyrl",
                           split="test", streaming=True,
                           revision=DATA_REVISIONS["fineweb2"])
    en_train = load_dataset("roneneldan/TinyStories", split="train", streaming=True,
                            revision=DATA_REVISIONS["tinystories"])
    en_eval = load_dataset("roneneldan/TinyStories", split="validation", streaming=True,
                           revision=DATA_REVISIONS["tinystories"])

    def take_text(dataset, limit):
        rows = []
        for row in dataset:
            rows.append(row["text"])
            if len(rows) >= limit:
                break
        return rows

    raw_train = take_text(ru_train, train_target) + take_text(en_train, train_target)
    raw_valid = take_text(ru_eval, eval_target) + take_text(en_eval, eval_target)
    train_docs, train_report = pipeline(raw_train, FilterConfig(), near_dup_threshold=0.85)
    valid_docs, valid_report = pipeline(raw_valid, FilterConfig(), near_dup_threshold=0.85)
    train_fingerprints = {hashlib.sha256(d.encode("utf-8")).digest() for d in train_docs}
    valid_docs = [d for d in valid_docs
                  if hashlib.sha256(d.encode("utf-8")).digest() not in train_fingerprints]
    print("train:", train_report)
    print("validation:", valid_report)
    print("revisions:", DATA_REVISIONS)
    print("\nAttribution: FineWeb2 (ODC-By 1.0 + Common Crawl Terms), "
          "TinyStories (CDLA-Sharing-1.0). This is technical documentation, not legal advice.")
else:
    DATA_REVISIONS = {"synthetic": f"generator-v1-seed-{SEED}"}
    print("USE_REAL_CORPUS = False — using the synthetic document-level split")

## 4. A byte-level BPE tokenizer

The initial alphabet consists of all 256 byte values, so valid UTF-8 text can be encoded without an unknown token. Ordinary Russian letters occupy two bytes before merging, but this does not describe every character in Unicode's Cyrillic blocks. BPE merges can join bytes within a character or span several characters.

Russian data volume matters, as do its diversity, vocabulary size, and preprocessing. We measure tokens per character and bytes per token on separate Russian and English probes. Read these as compression properties of this tokenizer on these examples, not as a direct measure of the language model's eventual quality.

In [ ]:
import time
from minillm.tokenizer import BBPETokenizer
from minillm.data import ByteTokenizer
from minillm.chat import CHAT_SPECIALS

# The chat control tokens are reserved in the vocabulary right away, even though
# pretraining does not need them. The reason is exactly the one behind the 1,075
# reserved identifiers in o200k_harmony (M2): adding them later means changing
# vocab_size and rebuilding the embeddings. Notebook 2 picks up this very
# checkpoint and expects to find `<|eot_id|>` inside it.
train_text = "\n".join(train_docs)
t0 = time.time()
tokenizer = BBPETokenizer.train(train_text, vocab_size=SETTINGS["vocab_size"],
                                specials=["<|bos|>", "<|eos|>"] + CHAT_SPECIALS)
print(f"trained in {time.time() - t0:.1f} s: vocabulary {tokenizer.vocab_size}, "
      f"{len(tokenizer.merges)} merges")
if tokenizer.vocab_size < SETTINGS["vocab_size"]:
    print('Available merge pairs ran out before the requested vocabulary size.')

sample = "The tokenizer predicts nothing; it only decides the price."
assert tokenizer.decode(tokenizer.encode(sample)) == sample, "round trip is broken"
print("round trip:", repr(tokenizer.decode(tokenizer.encode(sample))))

In [ ]:
probe_rng = random.Random(123)
ru_probe = " ".join(sentence(probe_rng, RU_SLOTS) for _ in range(40))
en_probe = " ".join(sentence(probe_rng, EN_SLOTS) for _ in range(40))
plain = ByteTokenizer()

rows = []
for name, probe in (("RU", ru_probe), ("EN", en_probe)):
    trained = tokenizer.fertility(probe)
    baseline = len(plain.encode(probe)) / len(probe)
    rows.append((name, trained["tokens_per_char"], baseline,
                 trained["bytes_per_token"]))

print(f"{'lang':<5} {'BBPE tok/char':>14} {'byte-level tok/char':>21} {'bytes/token':>12}")
for name, trained, baseline, bpt in rows:
    print(f"{name:<5} {trained:>14.3f} {baseline:>21.3f} {bpt:>12.2f}")
print("\nByte-level costs about two tokens per Cyrillic character — that is the "
      "price of having no merges for a language (M2).")

## 5. Sharding the token stream

Write the tokens into `.bin` shards and retain their checksums in `index.json`. Data volume, not the model profile alone, determines whether the corpus fits in memory. Small inputs let us inspect the reading order and integrity checks before scaling up. In particular, crossing a shard boundary must not discard the next-token transition.

In [ ]:
from minillm.data import ShardStream, TokenStream, write_shards

EOS = tokenizer.specials["<|eos|>"]

def encode_documents(documents):
    stream = []
    for doc in documents:
        stream.extend(tokenizer.encode(doc))
        stream.append(EOS)
    return stream

train_token_ids = encode_documents(train_docs)
valid_token_ids = encode_documents(valid_docs)
print(f"train tokens: {len(train_token_ids):,}; validation tokens: {len(valid_token_ids):,}")

shard_dir = Path("data_shards")
index = write_shards(train_token_ids, shard_dir / "train", shard_tokens=250_000)
write_shards(valid_token_ids, shard_dir / "valid", shard_tokens=250_000)

train_stream = ShardStream(shard_dir / "train", seq_len=SETTINGS["seq_len"], verify=True)
valid_stream = ShardStream(shard_dir / "valid", seq_len=SETTINGS["seq_len"], verify=True)
print(f"train: {len(index['shards'])} shards, {index['total_tokens']:,} tokens, "
      f"{len(train_stream):,} windows, checksum {train_stream.checksum()}")
print(f"validation: {len(valid_stream):,} windows — document-separated before tokenizer training")

## 6. Model config and budget

Before launching anything, three quantities the course insists on computing
before the run rather than after the OOM:

1. **parameters** — by the formula of M6, checked against the actual
   `model.parameters()`;
2. **KV cache per token** — `2·L·H_kv·d_head·sizeof(dtype)` (M5);
3. **the 20× Chinchilla-style reference budget** — 20 tokens per parameter (M9), and how many
   steps that is.

In [ ]:
from minillm.config import MiniLLMConfig, kv_bytes_per_token
from minillm.model import MiniLLM, MoELayer
from minillm.train import TrainConfig, memory_report

cfg = MiniLLMConfig.profile(
    SETTINGS["model_profile"],
    vocab_size=tokenizer.vocab_size,
    max_position_embeddings=max(SETTINGS["seq_len"], 256),
)
model = MiniLLM(cfg).to(DEVICE)

analytic = cfg.parameter_report()
assert analytic["total"] == model.num_parameters(), "formula and code disagree"
print(f"parameters: {analytic['total']:,} (formula == actual)")
for key in ("embeddings", "attention", "ffn", "norms"):
    print(f"  {key:<12} {analytic[key]:>12,}  ({analytic[key] / analytic['total']:.1%})")
print(f"  the block costs {analytic['block_over_d2']}·d²")
reference_cache_bpt = kv_bytes_per_token(cfg, dtype_bytes=2)
runtime_cache_bpt = kv_bytes_per_token(cfg, dtype_bytes={"fp32": 4, "bf16": 2, "fp16": 2}[COMPUTE_DTYPE])
print(f"\nKV cache, BF16/FP16 reference: {reference_cache_bpt} bytes/token "
      f"→ {reference_cache_bpt * SETTINGS['seq_len'] / 1024:.1f} KiB per window")
print(f"KV cache at the current compute dtype ({COMPUTE_DTYPE}): {runtime_cache_bpt} bytes/token")
print("persistent training memory:", memory_report(cfg, "adamw"))

In [ ]:
tcfg = TrainConfig(
    steps=SETTINGS["steps"],
    seq_len=SETTINGS["seq_len"],
    micro_batch_size=SETTINGS["micro_batch_size"],
    grad_accum=SETTINGS["grad_accum"],
    optimizer="muon",
    compute_dtype="auto",
    gradient_checkpointing=DEVICE.startswith("cuda"),
    seed=SEED,
    device=DEVICE,
    log_every=max(1, SETTINGS["steps"] // 10),
    out_dir="runs/pretrain",
)

chinchilla_tokens = 20 * analytic["total"]
planned_tokens = tcfg.steps * tcfg.tokens_per_step
print(f"planned next-token targets per step: {tcfg.tokens_per_step:,}")
print(f"planned:         {planned_tokens:,} supervised targets over {tcfg.steps:,} steps")
print(f"reference 20×: {chinchilla_tokens:,} tokens "
      f"→ {chinchilla_tokens / tcfg.tokens_per_step:,.0f} steps")
print(f"share of budget: {planned_tokens / chinchilla_tokens:.1%}")
print(f"epochs over the corpus: {planned_tokens / len(train_token_ids):.2f}")

The planned budget counts next-token targets: a `seq_len+1` window supplies `seq_len` targets. Runtime logs distinguish input positions (`tokens`, including the first context position and PAD) from supervised targets after shifting (`supervised_tokens`). Throughput, `tokens_per_s`, uses input positions. Training and validation loss average over supervised targets; gradient accumulation uses one denominator for the whole effective batch.

The budget fraction compares planned tokens with the $20P$ reference. It describes experiment size rather than diagnosing whether training is sufficient. The appropriate budget depends on the data and objective. `SMOKE` checks the pipeline with a short run; `FULL` sets a longer teaching schedule.

Epoch count records data reuse. Repeated exposure to a small corpus can encourage memorization, but there is no universal four-epoch boundary. Inspect both training and held-out curves, and remember that shared synthetic templates limit the diversity of both splits.

## 7. The run

The optimizer is a hybrid: Muon on two-dimensional matrices, AdamW on embeddings,
norms and the router (M9). The schedule is WSD — warmup, a plateau, a decay
(M9). The plateau is what lets a run be interrupted at any point and finished
with a decay tail, without retuning the schedule for a new length.

In [ ]:
from minillm.train import train, save_checkpoint

state = train(model, train_stream, tcfg, eval_stream=valid_stream)
print(f"\ndone: {state.step} steps, "
      f"{state.tokens:,} input positions, {state.supervised_tokens:,} targets, "
      f"final loss {state.history[-1]['loss']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

steps = [h["step"] for h in state.history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(steps, [h["loss"] for h in state.history], label="train", color="#5E81AC")
if "eval_loss" in state.history[0]:
    axes[0].plot(steps, [h["eval_loss"] for h in state.history], label="valid",
                 color="#BF616A")
axes[0].axhline(np.log(cfg.vocab_size), ls="--", lw=1, color="#4C566A",
                label="ln(V) — a uniform model")
axes[0].set_xlabel("step"); axes[0].set_ylabel("cross-entropy"); axes[0].legend()
axes[0].set_title("loss")

axes[1].plot(steps, [h["lr_scale"] for h in state.history], color="#A3BE8C")
axes[1].set_xlabel("step"); axes[1].set_ylabel("lr multiplier")
axes[1].set_title("WSD schedule")

axes[2].plot(steps, [h["tokens_per_s"] for h in state.history], color="#D08770")
axes[2].set_xlabel("step"); axes[2].set_ylabel("input positions/s")
axes[2].set_title("throughput")

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

bpc = state.history[-1]["loss"] / np.log(2)
print(f"final cross-entropy {state.history[-1]['loss']:.4f} nats "
      f"= {bpc:.4f} bits per token")
print('The curve describes this budget and corpus; a plateau alone does not identify the limiting cause.')

### 7.1. A sparse twin: total and active parameters

Replace the dense FFN with experts while keeping vocabulary, depth, attention, and positions fixed. Expert width targets a similar active-parameter budget, not exact equality. The standard S and S-MoE profiles have 6,327,552 and 6,593,792 active parameters respectively; the table below recomputes the counts for this run's vocabulary.

Similar counts do not promise equal token latency. Routing, matrix shapes, and output combination also cost work. KV geometry does remain identical when attention is unchanged. Begin by verifying those distinct statements numerically.

In [ ]:
dense_cfg = cfg
moe_cfg = MiniLLMConfig(**{**MiniLLMConfig.twin(SETTINGS["model_profile"]).to_dict(),
                           "vocab_size": tokenizer.vocab_size,
                           "max_position_embeddings": cfg.max_position_embeddings})
moe_model = MiniLLM(moe_cfg).to(DEVICE)

rows = [("dense", dense_cfg, model), ("MoE", moe_cfg, moe_model)]
print(f"{'variant':<8} {'total':>14} {'active':>12} {'sparsity':>9} "
      f"{'KV B/tok':>9} {'FLOP/tok':>12}")
for name, c, m in rows:
    assert c.parameter_report()["total"] == m.num_parameters(), "formula diverged"
    total, active = c.parameter_report()["total"], c.active_parameters()
    print(f"{name:<8} {total:>14,} {active:>12,} {total / active:>8.2f}× "
          f"{kv_bytes_per_token(c):>9,} {c.flops_per_token():>12,}")

print(f"\n{moe_cfg.num_local_experts} experts, top-{moe_cfg.num_experts_per_tok}, "
      f"{moe_cfg.num_shared_experts} shared, first layer dense: "
      f"{not moe_cfg.moe_layer(0)}")
print(f"MoE active relative to dense: "
      f"×{moe_cfg.active_parameters() / dense_cfg.active_parameters():.3f}")
print(f"KV cache matches to the byte: "
      f"{kv_bytes_per_token(dense_cfg) == kv_bytes_per_token(moe_cfg)} — "
      f"attention untouched, so the comparison is clean")

The table separates bank capacity from the work of selected experts. Train both models for the same step and token budget, then compare loss, throughput, and assignments. Neither an MoE quality gain nor equal hardware cost is assumed in advance.

In [ ]:
moe_state = train(moe_model, train_stream, tcfg, eval_stream=valid_stream)

dense_last, moe_last = state.history[-1], moe_state.history[-1]
print(f"\n{'variant':<8} {'train':>9} {'valid':>9} {'input/s':>9}")
for name, h in (("dense", dense_last), ("MoE", moe_last)):
    print(f"{name:<8} {h['loss']:>9.4f} {h.get('eval_loss', float('nan')):>9.4f} "
          f"{h['tokens_per_s']:>9.0f}")

moe_layer = next(l.ffn for l in moe_model.layers if isinstance(l.ffn, MoELayer))
load = moe_layer.expert_load
print(f"\nexpert load at the last measurement: min {load.min():.3f}, "
      f"max {load.max():.3f}, std {load.std():.3f}")
print(f"a perfectly even load would be "
      f"{moe_cfg.num_experts_per_tok / moe_cfg.num_local_experts:.3f} per expert")
print(f"aux-loss-free bias: max|b| = {moe_layer.expert_bias.abs().max():.4f}")

**Interpreting the comparison.** Uniform routing would give each of $E$ experts an average fraction $k/E$ of tokens. Actual assignments may be uneven, and fewer tokens do not automatically mean fewer optimizer steps. Inspect the load table to see how much data each expert receives.

One short run compares two particular configurations. Parameter counts alone cannot explain a loss gap. Notebook 4 isolates a balancing-rate change and measures dropped assignments, providing additional evidence about routing without assuming its effect on quality.

## 8. Inspecting the trained model

Generate continuations with greedy decoding and sampling. Prefill processes prompt positions; decode continues using the KV cache. Module 10 explains when compute or memory traffic becomes limiting, but those conditions cannot be assumed for this small CPU run. Record the measured times and inspect the text, separating a working generation path from useful language quality.

In [ ]:
from minillm.generate import SamplingConfig, generate

for temperature, top_p in ((0.0, 1.0), (0.8, 0.9)):
    prompt = tokenizer.encode("a language model")
    out, stats = generate(model, prompt, 80,
                          SamplingConfig(temperature=temperature, top_p=top_p, seed=SEED),
                          stop_tokens={EOS})
    label = "greedy" if temperature == 0 else f"T={temperature}, top_p={top_p}"
    print(f"--- {label}\n{tokenizer.decode(out)!r}\n")
print("stats of the last run:", stats.as_dict())

## 9. A passport for our own model

The course opened by teaching us to read someone else's `config.json`. Now the
model is ours — and the same passport approach reads it with no changes at all.
That is the ring closing: the arithmetic of modules 1, 5 and 6 applied to a file
we just wrote ourselves.

In [ ]:
run_dir = Path("runs/pretrain/final")
save_checkpoint(run_dir, model, state.optimizer, tcfg, state)
tokenizer.save(run_dir / "tokenizer.json")
print("saved:", sorted(p.name for p in run_dir.iterdir()))

In [ ]:
import json

def read_passport(path: Path) -> None:
    """A model passport from config.json — the same move as in module 1."""
    c = json.loads((path / "config.json").read_text(encoding="utf-8"))
    heads, kv = c["num_attention_heads"], c["num_key_value_heads"]
    attention = "MHA" if heads == kv else ("MQA" if kv == 1 else f"GQA {heads}/{kv}")
    ffn_ratio = c["intermediate_size"] / c["hidden_size"]
    cache_bytes = {"fp32": 4, "bf16": 2, "fp16": 2}[stats.cache_dtype]
    kv_bytes = 2 * c["num_hidden_layers"] * kv * c["head_dim"] * cache_bytes

    print("Axis I   · size      :", f"d={c['hidden_size']}, L={c['num_hidden_layers']}, "
          f"V={c['vocab_size']}, embeddings "
          f"{'tied' if c['tie_word_embeddings'] else 'untied'}")
    print("Axis II  · context   :", f"{c['max_position_embeddings']} tokens, "
          f"RoPE base {c['rope_theta']:g}, scaling "
          f"{c['rope_scaling'] or 'none'}")
    print("Axis III · attention :", attention,
          f"· head_dim {c['head_dim']} · sinks {'yes' if c['attention_sinks'] else 'no'}"
          f" · window {c['sliding_window'] or 'none'}")
    print("Axis IV  · block     :", f"{c['norm_placement']}-norm, {c['hidden_act']}, "
          f"d_ffn/d = {ffn_ratio:.2f}"
          + (f", clamp {c['swiglu_limit']}" if c["swiglu_limit"] else ""))
    print("Axis V   · sparsity  :", "dense" if not c["num_local_experts"]
          else f"MoE, {c['num_local_experts']} experts, top-{c['num_experts_per_tok']}")
    print("Derived              :", f"KV cache {kv_bytes} B/token in measured {stats.cache_dtype}, "
          f"{kv_bytes * c['max_position_embeddings'] / 1024 ** 2:.1f} MiB at full context")

read_passport(run_dir)

## 10. Summary and what comes next

We walked the road the course took apart across the course modules: raw documents
→ cleaning funnel → our own BPE → shards → the course reference model → a run with Muon
and WSD → generation → a passport. Every number was computed here; none was taken
from the lectures on faith.

The passport computes the training budget relative to the $20P$ reference.
That ratio depends on the profile and step count; it is not a quality score.
The synthetic corpus lets us trace learning and check the handoffs between
stages. Architecture comparisons require a controlled experiment such as those
in Notebook 4. Claims about natural language need suitable data and response
evaluation in addition to a successful training run.

**Next:**

- notebook 2 — post-training: chat template, loss masking over assistant spans,
  packing and DPO;
- notebook 3 — inference: prefix caching, speculative decoding with a small
  draft model, and quantization;
- notebook 4 — ablations: the course's claims checked by number, plus an honest
  account of where this scale runs out.

**Data attribution.** The synthetic corpus of §2 is generated by this notebook's
own code. With `USE_REAL_CORPUS` enabled, the licences of the respective datasets
apply: FineWeb-2 under ODC-By 1.0 plus the CommonCrawl terms of use, TinyStories
under CDLA-Sharing-1.0.